In [6]:
# === One-cell image evaluation (REAL=0, FAKE=1) ===
import os, glob
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------
# 1) Paths & auto-detect
# -----------------------
# Weight file name you saved earlier
WEIGHTS = "best_model.pth"

# Candidate folders where your val images might be (pick first that exists)
CANDIDATE_REAL = [
    "data/cropped/val/real",
    "data/processed/val/real",
]
CANDIDATE_FAKE = [
    "data/cropped/val/fake",
    "data/processed/val/fake",
]

def first_existing(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    return None

DIR_REAL = first_existing(CANDIDATE_REAL)
DIR_FAKE = first_existing(CANDIDATE_FAKE)

print("CWD:", os.getcwd())
print("Using REAL dir:", DIR_REAL)
print("Using FAKE dir:", DIR_FAKE)
assert DIR_REAL and DIR_FAKE, "Could not find val image folders. Check your paths."

# -----------------------
# 2) Model definition
# -----------------------
# If SimpleCNN is already defined in your notebook, this will be skipped
try:
    SimpleCNN  # type: ignore
except NameError:
    class SimpleCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
            self.pool  = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
            self.fc1   = nn.Linear(32 * 56 * 56, 128)  # for 224x224 input after two pools
            self.fc2   = nn.Linear(128, 2)

        def forward(self, x):
            x = self.pool(F.relu(self.conv1(x)))  # -> [B,16,112,112]
            x = self.pool(F.relu(self.conv2(x)))  # -> [B,32, 56, 56]
            x = x.view(-1, 32 * 56 * 56)
            x = F.relu(self.fc1(x))
            return self.fc2(x)

# -----------------------
# 3) Load model weights
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
assert os.path.exists(WEIGHTS), f"Weight file not found: {WEIGHTS}"
model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()
print("Loaded weights from:", WEIGHTS)

# -----------------------
# 4) Collect files
# -----------------------
def list_images(d):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        files += glob.glob(os.path.join(d, "**", ext), recursive=True)
    return sorted(files)

files_real = list_images(DIR_REAL)
files_fake = list_images(DIR_FAKE)
print("Found files  -> REAL:", len(files_real), " FAKE:", len(files_fake))
assert (len(files_real) + len(files_fake)) > 0, "No images found to evaluate."

# -----------------------
# 5) Transforms (match training)
# -----------------------
tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# -----------------------
# 6) Eval loop
# -----------------------
y_true, y_pred = [], []

@torch.no_grad()
def run_dir(files, label):
    for fp in files:
        try:
            img = Image.open(fp).convert("RGB")
        except Exception:
            continue  # skip unreadable files
        x = tfm(img).unsqueeze(0).to(device)
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()
        y_true.append(label)
        y_pred.append(pred)

run_dir(files_real, 0)  # REAL -> 0
run_dir(files_fake, 1)  # FAKE -> 1

# -----------------------
# 7) Sanity + metrics
# -----------------------
print("Lens:", len(y_true), len(y_pred))
print("y_true dist:", Counter(y_true))
print("y_pred dist:", Counter(y_pred))
assert len(y_true) > 0 and len(y_true) == len(y_pred), "No eval data or mismatch."

labels = [0, 1]
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred, labels=labels))

print("\nClassification report:")
print(classification_report(y_true, y_pred,
                            labels=labels,
                            target_names=["REAL", "FAKE"],
                            zero_division=0))


CWD: /Users/apple/Desktop/revealed
Using REAL dir: data/cropped/val/real
Using FAKE dir: data/cropped/val/fake
Loaded weights from: best_model.pth
Found files  -> REAL: 2875  FAKE: 13716
Lens: 16591 16591
y_true dist: Counter({1: 13716, 0: 2875})
y_pred dist: Counter({1: 13666, 0: 2925})

Confusion matrix (rows=true, cols=pred):
[[ 2848    27]
 [   77 13639]]

Classification report:
              precision    recall  f1-score   support

        REAL       0.97      0.99      0.98      2875
        FAKE       1.00      0.99      1.00     13716

    accuracy                           0.99     16591
   macro avg       0.99      0.99      0.99     16591
weighted avg       0.99      0.99      0.99     16591

